This notebook is for test bot on local CPU.

In [1]:
import os
import io
import time
import queue
import threading
from PIL import Image
from dotenv import load_dotenv
import telebot
from telebot import types


import torch
from pillow_heif import register_heif_opener

In [2]:
# HEIC support
register_heif_opener()

In [ ]:
TOKEN = "YOUR_TOKEN"
bot = telebot.TeleBot(TOKEN)

In [ ]:

device = torch.device("cpu")
model = model.py
model.to(device).eval()

In [85]:
# ======================
# USER STATE
# ======================
user_mode = {} # chat_id -> mode ("bg" or "convert")
user_convert_format = {} # chat_id -> "PNG" or "JPG"
user_bg_mode = {}       # transparent / white / custom
user_bg_color = {} 

In [86]:
task_queue = queue.Queue()

In [ ]:
def get_pil_image_from_message(message):
    # If it's a photo
    if message.content_type == "photo":
        file_id = message.photo[-1].file_id

    # if it's a document
    elif message.content_type == "document":
        file_id = message.document.file_id

    else:
        raise ValueError("Неподдерживаемый тип сообщения")

    file_info = bot.get_file(file_id)
    downloaded_file = bot.download_file(file_info.file_path)

    return Image.open(io.BytesIO(downloaded_file))

In [ ]:
def convert_image_format(pil_image, target_format="PNG", jpg_quality=95):
    target_format = target_format.upper()


    if target_format not in ["PNG", "JPG", "JPEG"]:
        raise ValueError("Поддерживаются только PNG и JPG")

    output = io.BytesIO()

    if target_format in ["JPG", "JPEG"]:
    # JPG without transparency
        if pil_image.mode in ("RGBA", "LA", "P"):
            bg = Image.new("RGB", pil_image.size, (255, 255, 255))
            if pil_image.mode == "RGBA":
                bg.paste(pil_image, mask=pil_image.split()[-1])
            else:
                bg.paste(pil_image)
                pil_image = bg
        else:
            pil_image = pil_image.convert("RGB")


        pil_image.save(output, format="JPEG", quality=jpg_quality, optimize=True)


    else:
        if pil_image.mode not in ("RGB", "RGBA"):
            pil_image = pil_image.convert("RGBA")
        pil_image.save(output, format="PNG", optimize=True, compress_level=9)


    output.seek(0)
    return output

In [89]:
def resize_long_side(pil_image, max_side = 2000):
    """
    Уменбшает изображение так, чтобы длинная сторона была <= max_side.
    Пропорции сохраняются.
    """
    w, h = pil_image.size
    long_side = max(w,h)
    if long_side <= max_side:
        return pil_image
    scale = max_side/long_side
    new_size = (int(w*scale), int(h*scale))
    
    return pil_image.resize(new_size, Image.LANCZOS)

In [ ]:
def worker_thread():
    while True:
        task = task_queue.get()
        chat_id = task["chat_id"]
        message = task["message"]
        mode = task["mode"]
        target_format = task.get("format")
        try:
            start_time = time.time()
            pil_image = get_pil_image_from_message(message)


# ---------- BG REMOVE ----------
            if mode == "bg":
                src = pil_image.convert('RGB')
                result = model.inference(src, refine_foreground=False)
                
                if isinstance(result, tuple):
                    result_image = result[0]
                else:
                    result_image = result
                if result_image.mode != "RGBA":
                    result_image = result_image.convert("RGBA")
                    
                bg_mode = user_bg_mode.get(chat_id, "white")
                
                # ---------- transparent PNG ----------
                if bg_mode == "transparent":
                    output = io.BytesIO()
                    result_image.save(output, format="PNG")
                    output.seek(0)

                    bot.send_document(chat_id,output,visible_file_name="no_bg.png",
                                      timeout=120)
                
                elif bg_mode == "white":
                    bg = Image.new("RGB", result_image.size, (255, 255, 255))
                    bg.paste(result_image, mask=result_image.split()[3])
                    output = io.BytesIO()
                    bg.save(output, format="JPEG", quality=95)
                    output.seek(0)

                    bot.send_photo(chat_id, output, timeout=120)
                
                 # ---------- CUSTOM RGB ----------
                elif bg_mode == "custom":
                    color = user_bg_color.get(chat_id, (255, 255, 255))
                    bg = Image.new("RGB", result_image.size, color)
                    bg.paste(result_image, mask=result_image.split()[3])
                    output = io.BytesIO()
                    bg.save(output, format="JPEG", quality=95)
                    output.seek(0)
                    
                    bot.send_photo(chat_id, output, timeout=120)
                
                output.close()
                del result_image

# ---------- CONVERT ----------
            elif mode == "convert":
                pil_image = resize_long_side(pil_image, 2000)
                output_buffer = convert_image_format(pil_image, target_format)
                ext = "jpg" if target_format == "JPG" else "png"
                filename = f"converted.{ext}"
                bot.send_document(chat_id, output_buffer, visible_file_name=filename, timeout = 120)
                output_buffer.close()


            else:
                raise ValueError("Неизвестный режим обработки")


            #bot.send_document(chat_id, output_buffer, visible_file_name=filename)


            elapsed = time.time() - start_time
            m = int(elapsed // 60)
            s = int(elapsed % 60)
            bot.send_message(chat_id, f"Готово! ⏱ {m} мин {s} сек.")


            #output_buffer.close()
            del pil_image


        except Exception as e:
            bot.send_message(chat_id, f"Ошибка обработки: {e}")


        finally:
            task_queue.task_done()

threading.Thread(target=worker_thread, daemon=True).start()

In [91]:
# ======================
# MENUS
# ======================

def main_menu():
    kb = types.ReplyKeyboardMarkup(resize_keyboard=True)
    kb.add("Изменение фона", "Конвертер")
    return kb

def convert_menu():
    kb = types.ReplyKeyboardMarkup(resize_keyboard=True)
    kb.add("PNG", "JPG")
    kb.add("⬅️ Назад")
    return kb

def bg_menu():
    kb = types.ReplyKeyboardMarkup(resize_keyboard=True)
    kb.add("PNG без фона")
    kb.add("Белый фон")
    kb.add("Цвет фона по RGB")
    kb.add("⬅️ Назад")
    return kb

In [92]:
import re

def parse_rgb(text):
    match = re.search(r"\(?\s*(\d{1,3})\s*,\s*(\d{1,3})\s*,\s*(\d{1,3})\s*\)?", text)
    if not match:
        return None

    r, g, b = map(int, match.groups())

    if any(x > 255 for x in (r, g, b)):
        return None

    return (r, g, b)

In [ ]:
# ---------------------- #
#       BOT COMMANDS
# ---------------------- #
@bot.message_handler(commands=["start"])
def start_cmd(message):
    bot.send_message(
        message.chat.id,
        "Привет! Я ваш бот для работы с изображениями. Вот, что я могу: удалять фон, перекрашивать его в белый или в нужный Вам цвет, а также конвертировать изображения из форматов png, jpg, jpeg, webp, heic и heif в png или jpg.👋 Выберите режим работы:",
        reply_markup=main_menu()
        )
    
# ======================
# MENU HANDLERS
# ======================

@bot.message_handler(func=lambda m: m.text == "Изменение фона")
def choose_bg(message):
    user_mode[message.chat.id] = "bg"
    bot.send_message(
        message.chat.id,
        "Выберите вариант результата:",
        reply_markup=bg_menu()
    )
    
@bot.message_handler(func=lambda m: m.text == "PNG без фона")
def bg_transparent(message):
    user_bg_mode[message.chat.id] = "transparent"
    bot.send_message(message.chat.id, "Пришлите изображение.")


@bot.message_handler(func=lambda m: m.text == "Белый фон")
def bg_white(message):
    user_bg_mode[message.chat.id] = "white"
    bot.send_message(message.chat.id, "Пришлите изображение.")


@bot.message_handler(func=lambda m: m.text == "Цвет фона по RGB")
def bg_custom(message):
    user_bg_mode[message.chat.id] = "custom"
    bot.send_message(
        message.chat.id,
        "Пришлите цвет в формате (R, G, B)\nНапример: (222, 218, 47)"
    ) 
    
@bot.message_handler(func=lambda m: user_bg_mode.get(m.chat.id) == "custom")
def receive_rgb(message):
    rgb = parse_rgb(message.text)

    if not rgb:
        bot.send_message(
            message.chat.id,
            "Неверный формат. Пример: (120, 200, 30)"
        )
        return

    user_bg_color[message.chat.id] = rgb
    bot.send_message(
        message.chat.id,
        f"Цвет принят {rgb}. Пришлите изображение."
    )
    
@bot.message_handler(func=lambda m: m.text == "Конвертер")
def choose_convert(message):
    user_mode[message.chat.id] = "convert"
    bot.send_message(
        message.chat.id,
        "Выберите формат результата:",
        reply_markup=convert_menu()
    )

@bot.message_handler(func=lambda m: m.text in ["PNG", "JPG"])
def choose_format(message):
    if user_mode.get(message.chat.id) != "convert":
        return

    user_convert_format[message.chat.id] = message.text
    bot.send_message(
        message.chat.id,
        f"Пришлите изображение — конвертирую в {message.text}."
    )

@bot.message_handler(func=lambda m: m.text == "⬅️ Назад")
def back_to_main(message):
    bot.send_message(
        message.chat.id,
        "Главное меню:",
        reply_markup=main_menu()
    )
# ======================
#   IMAGE HANDLERS
# ======================

allowed_mime_types = [
    "image/png",
    "image/jpeg",
    "image/jpg",
    "image/webp",
    "image/heic",
    "image/heif"
]

@bot.message_handler(content_types=["photo", "document"])
def handle_image(message):
    chat_id = message.chat.id
    mode = user_mode.get(chat_id)

    if not mode:
        bot.send_message(chat_id, "Сначала выберите режим через /start")
        return

    if message.content_type == "document":
        if message.document.mime_type not in allowed_mime_types:
            bot.send_message(chat_id, "❌ Это не изображение")
            return

    # ----- BG REMOVE -----
    if mode == "bg":
        task_queue.put({
            "chat_id": chat_id,
            "message": message,
            "mode": "bg"
        })

    # ----- CONVERT -----
    elif mode == "convert":
        fmt = user_convert_format.get(chat_id)
        if not fmt:
            bot.send_message(chat_id, "Сначала выберите PNG или JPG")
            return

        task_queue.put({
            "chat_id": chat_id,
            "message": message,
            "mode": "convert",
            "format": fmt
        })

    bot.send_message(
        chat_id,
        f"Файл получен 📥 Место в очереди: {task_queue.qsize()}"
    )


In [ ]:
#loading bot
if __name__ == "__main__":
    print("Бот запущен.")
    bot.infinity_polling(skip_pending=True, timeout=20, long_polling_timeout=20)

Бот запущен.


2026-02-09 22:59:41,760 (__init__.py:1121 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
2026-02-09 22:59:41,762 (__init__.py:1123 MainThread) ERROR - TeleBot: "Break infinity polling"
